In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit\\')

In [ ]:
PrevLoans = pd.read_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\prev_loan_data_processed.parquet")
PrevLoans = PrevLoans.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

In [5]:
POSBal = pd.read_csv('data/Internal/POS_CASH_balance.csv')
POSBal = POSBal.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})
POSBal.sort_values(by=['SK_ID_CURR','SK_ID_PREV','MONTHS_BALANCE'], inplace=True, ignore_index=True)

In [6]:
base = pd.read_csv('data/Internal/application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

In [7]:
POSBal = base.merge(POSBal, on='SK_ID_CURR', how='inner').drop(columns=['TARGET'])

Impute missing data

In [8]:
POSBal['CNT_INSTALMENT'] = POSBal.groupby(['SK_ID_PREV'])['CNT_INSTALMENT'].transform(
    lambda x: x.ffill(limit=3).bfill(limit=3)
)
POSBal['CNT_INSTALMENT_FUTURE'] = POSBal.groupby(['SK_ID_PREV'])['CNT_INSTALMENT_FUTURE'].transform(
    lambda x: x.ffill(limit=3).bfill(limit=3)
)

In [9]:
#check for installment prepayment

Aggregate to Monthly

In [10]:
POSBal_Mon = POSBal.groupby(['SK_ID_CURR','MONTHS_BALANCE'])[['CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE', 'SK_DPD', 'SK_DPD_DEF']].agg('max')

In [11]:
POSBal_Mon.columns = [''.join(x).upper() for x in POSBal_Mon.columns]
POSBal_Mon = POSBal_Mon.reset_index()

Aggegate data at SK_ID_CURR

In [12]:
pos_aggregated = POSBal_Mon.groupby('SK_ID_CURR').agg(
    POS_DAYS_IN_DPD=('SK_DPD', lambda x: sum((x > 0)*1)),
    POS_DAYS_IN_DPD_DEF=('SK_DPD_DEF', lambda x: sum((x > 0)*1)),
    POS_DAYS_IN_DPD_TO_DURATION_RATIO=('SK_DPD', lambda x: sum((x > 0)*1)/len(x) if len(x) > 0 else 0),
)

In [13]:
pos_aggregated_1Y = POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -12].groupby('SK_ID_CURR').agg(
    POS_DAYS_IN_DPD_1Y=('SK_DPD', lambda x: sum((x > 0)*1)),
    POS_DAYS_IN_DPD_DEF_1Y=('SK_DPD_DEF', lambda x: sum((x > 0)*1)),
    POS_DAYS_IN_DPD_TO_DURATION_RATIO_1Y=('SK_DPD', lambda x: sum((x > 0)*1)/len(x) if len(x) > 0 else 0),
)

In [14]:
pos_aggregated_3M = POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -3].groupby('SK_ID_CURR').agg(
    POS_DAYS_IN_DPD_3M=('SK_DPD', lambda x: sum((x > 0)*1)),
    POS_DAYS_IN_DPD_DEF_3M=('SK_DPD_DEF', lambda x: sum((x > 0)*1)),
    POS_DAYS_IN_DPD_TO_DURATION_RATIO_3M=('SK_DPD', lambda x: sum((x > 0)*1)/len(x) if len(x) > 0 else 0),
)

Prepayment Flag

In [42]:
POSBal_first = POSBal.groupby(['SK_ID_CURR','SK_ID_PREV']).first().reset_index()
POSBal_last = POSBal.groupby(['SK_ID_CURR','SK_ID_PREV']).last().reset_index()

Prepayment Flag is actual duration < originalexpected duration

In [ ]:
POSBal_CNT_INSTALMENT = (POSBal_first[[ 'SK_ID_CURR', 'SK_ID_PREV', 'CNT_INSTALMENT']]
           .merge(POSBal_last[[ 'SK_ID_CURR', 'SK_ID_PREV', 'CNT_INSTALMENT']], on=['SK_ID_CURR','SK_ID_PREV'], suffixes=('_FIRST', '_LAST'))
    )
POSBal_CNT_INSTALMENT['PREV_CREDIT_PREPAYMENT_FLAG'] = (POSBal_CNT_INSTALMENT['CNT_INSTALMENT_FIRST'] != POSBal_CNT_INSTALMENT['CNT_INSTALMENT_LAST'])*1

In [ ]:
POSBal_Prepay_flag = POSBal_CNT_INSTALMENT.groupby('SK_ID_CURR')['PREV_CREDIT_PREPAYMENT_FLAG'].max().reset_index()

Aggregate last month values

In [47]:
POSBal_Agg_last = POSBal_last.groupby('SK_ID_CURR').agg(
    POS_AVG_CNT_INSTALMENT_LAST_MONTH=('CNT_INSTALMENT', 'mean'),
    POS_AVG_CNT_INSTALMENT_FUTURE_LAST_MONTH=('CNT_INSTALMENT_FUTURE', 'mean'),
    POS_MAX_CNT_INSTALMENT_LAST_MONTH=('CNT_INSTALMENT', 'max'),
    POS_MAX_CNT_INSTALMENT_FUTURE_LAST_MONTH=('CNT_INSTALMENT_FUTURE', 'max'),
    POS_CNT_ACTIVE_PREV_CONTRACT_STATUS_LAST_MONTH=('NAME_CONTRACT_STATUS', lambda x: sum((x == 'Active')*1)),
    POS_SK_DPD_LAST_MONTH=('SK_DPD', 'max'),
    POS_SK_DPD_DEF_LAST_MONTH=('SK_DPD_DEF', 'max')
)

In [48]:
POSBal_latest_dpd = POSBal.groupby('SK_ID_CURR')['MONTHS_BALANCE'].max().reset_index()
POSBal_latest_dpd['POS_DAYS_SINCE_LAST_DPD_GT_1'] = - POSBal_latest_dpd['MONTHS_BALANCE']
POSBal_latest_dpd.drop(columns=['MONTHS_BALANCE'], inplace=True)    

Aggregate at Previous ID level

In [49]:
POSBal_PrevLoans = POSBal_CNT_INSTALMENT.merge(PrevLoans, on=['SK_ID_PREV', 'SK_ID_CURR'], how='left')

NameError: name 'PrevLoans' is not defined

In [21]:
POSBal_PrevLoans['AMT_ANNUITY_TO_CREDIT_RATIO'] = POSBal_PrevLoans['AMT_ANNUITY'] / POSBal_PrevLoans['AMT_CREDIT']

In [22]:
POSBal_Agg_PrevLoans = POSBal_PrevLoans.groupby('SK_ID_CURR').agg(
    POS_TOTAL_PREV_LOAN = ('SK_ID_PREV', 'nunique'),   
    POS_SUM_PREV_CREDIT_PREPAYMENT_FLAG = ('PREV_CREDIT_PREPAYMENT_FLAG', 'sum'),

    POS_AVG_CREDIT_TO_APPL = ('CREDIT_TO_APPL', 'mean'),
    POS_ANNUITY_TO_CREDIT_RATIO = ('AMT_ANNUITY_TO_CREDIT_RATIO', 'mean'),

    POS_CNT_HIGH_YIELD_GROUP = ('NAME_YIELD_GROUP', lambda x: ((x=='high')*1).sum()),
    POS_CNT_MIDDLE_YIELD_GROUP = ('NAME_YIELD_GROUP', lambda x: ((x=='middle')*1).sum()),
    POS_CNT_NFLAG_INSURED_ON_APPROVAL_PREVLOAN = ('NFLAG_INSURED_ON_APPROVAL', 'sum'),
    POS_AVG_DAYS_DECISION_PREVLOAN = ('DAYS_DECISION', 'mean')
)

In [23]:
del PrevLoans, POSBal_Mon, POSBal_first, POSBal_last, POSBal_CNT_INSTALMENT

Merge all  

In [ ]:
final_pos_aggregated = pos_aggregated.copy()

# Merge 1Y features
final_pos_aggregated = final_pos_aggregated.merge(
    pos_aggregated_1Y, 
    on='SK_ID_CURR', 
    how='left',
    indicator=True
)
# 'both' means the ID was found in both, 'left_only' means it was missing in the right side
final_pos_aggregated['POS_IS_IN_LAST_1Y'] = final_pos_aggregated['_merge'].apply(
    lambda x: 1 if x == 'both' else 0
)
final_pos_aggregated.drop(columns=['_merge'], inplace=True)

# Merge 3M features
final_pos_aggregated = final_pos_aggregated.merge(
    pos_aggregated_3M, 
    on='SK_ID_CURR', 
    how='left',
    indicator=True
)
final_pos_aggregated['POS_IS_IN_LAST_3M'] = final_pos_aggregated['_merge'].apply(
    lambda x: 1 if x == 'both' else 0
)
final_pos_aggregated.drop(columns=['_merge'], inplace=True)


# Merge Last Month Contract Status/Installment features
final_pos_aggregated = final_pos_aggregated.merge(
    POSBal_Agg_last, 
    on='SK_ID_CURR', 
    how='left'
)
# Merge days to last DPD features
final_pos_aggregated = final_pos_aggregated.merge(
    POSBal_latest_dpd, 
    on='SK_ID_CURR', 
    how='left'
)

# Merge Previous Loan features fromm Previous Loan application data
final_pos_aggregated = final_pos_aggregated.merge(
    POSBal_Agg_PrevLoans, 
    on='SK_ID_CURR', 
    how='left'
)

# Merge base and target
final_pos_aggregated = final_pos_aggregated.merge(
    base, 
    on='SK_ID_CURR', 
    how='left'
)

# Check the result
print(f"Final shape: {final_pos_aggregated.shape}")
final_pos_aggregated.head()

Final shape: (289444, 29)


,SK_ID_CURR,POS_DAYS_IN_DPD,POS_DAYS_IN_DPD_DEF,POS_DAYS_IN_DPD_TO_DURATION_RATIO,POS_DAYS_IN_DPD_1Y,POS_DAYS_IN_DPD_DEF_1Y,POS_DAYS_IN_DPD_TO_DURATION_RATIO_1Y,POS_IS_IN_LAST_1Y,POS_DAYS_IN_DPD_3M,POS_DAYS_IN_DPD_DEF_3M,...,MONTHS_BALANCE,POS_TOTAL_PREV_LOAN,POS_SUM_PREV_CREDIT_PREPAYMENT_FLAG,POS_AVG_CREDIT_TO_APPL,POS_ANNUITY_TO_CREDIT_RATIO,POS_CNT_HIGH_YIELD_GROUP,POS_CNT_MIDDLE_YIELD_GROUP,POS_CNT_NFLAG_INSURED_ON_APPROVAL_PREVLOAN,POS_AVG_DAYS_DECISION_PREVLOAN,TARGET
0,100002,0,0,0.0,0.0,0.0,0.0,1,0.0,0.0,...,-1,1,0,1.000000,0.051670,0,0,0.0,-606.000000,1
1,100003,0,0,0.0,NaN,NaN,NaN,0,NaN,NaN,...,-18,3,1,1.057664,0.126383,0,2,2.0,-1305.000000,0
2,100004,0,0,0.0,NaN,NaN,NaN,0,NaN,NaN,...,-24,1,1,0.828021,0.266450,0,1,0.0,-815.000000,0
3,100006,0,0,0.0,0.0,0.0,0.0,1,0.0,0.0,...,-1,3,2,0.899970,0.082260,1,1,0.0,-369.666667,0
4,100007,0,0,0.0,0.0,0.0,0.0,1,0.0,0.0,...,-1,5,1,1.002827,0.099641,3,2,3.0,-1294.000000,0


In [ ]:
del pos_aggregated, pos_aggregated_1Y, pos_aggregated_3M, POSBal_Agg_last, POSBal_latest_dpd, POSBal_Agg_PrevLoans, base, POSBal_PrevLoans

In [37]:
final_pos_aggregated.rename(columns={'MONTHS_BALANCE': 'POS_DAYS_SINCE_LAST_DPD_GT_1'}, inplace=True)

In [38]:
final_pos_aggregated.to_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\posdata_processed.parquet")